# MiniMax H3 エピソード一発（選んで Run all）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/h3-kasumi-adult-0402/minimax_h3_episode_bot.ipynb)

**コードセルは1本。迷ったらドロップダウンはそのままで Run all。** Drive `minimax-h3-comfyui/episodes/<slug>/` に
`episode.json` とスチールが無ければ GitHub から取ってくる。全ビートを1つのランタイムで描き、
HUD・タイトル・免責エンドカードを載せて `final/<slug>-<日時>.mp4`（と `latest.mp4`）を書く。終わったら停止。

## 上から 3 つだけ選ぶ（迷ったらそのまま）

**1. つなぎ方** — 動画をどう繋げるか

- **カット（本ごと独立・迷ったらこれ）** … プロンプトで直したい。カメラも変えられる
- **前の最終フレームから続ける** … 2本目以降を前クリップの最後のコマから I2V。つながり優先
- **用意した最終フレームへ着く** … stills の jpg を最後のコマにする。jpg がある話だけ

**2. カメラ**

- **横スク（真横・全身・迷ったらこれ）** … 常に真横の 2D 横スクロール
- **3Dアクション（引きの三人称）** … 引きの三人称。カットつなぎのとき画角が回る

**3. 画質**

- **スピード（最速）** … 試し打ち。turbo 4step。格闘 LoRA なし
- **バランス（迷ったらこれ）** … 普段使い。Larry 8step
- **質（きれい・時間かかる）** … きれい優先。Larry 12step

シネマ LoRA は積まない。スローモーションの語は書かない。視点は三人称ゲームのまま。

- 本番の inbox / queued / output は触らない。`models/` だけ共有
- あさの 10Eros Max は Drive `models/diffusion_models/10Eros_Max_h3_TURBO-hybrid_beta5_int8.safetensors` を使う（HuggingFace からは取らない）
- 途中で止まっても `raw/<beat>.mp4` があるビートは飛ばして再開（FRESH で作り直し）
- HUD・字幕は生成後に載せる。H3 に日本語UIを描かせない
- 投稿しない。アフィURL禁止。他のネタは `minimaxh3/episodes/_template` を複製して EPISODE を変える
- `EPISODE = "kasumi-late-desk-adult"` は霞東あさ。Combat は 06 と 10。マージ前は `BRANCH` もこの PR ブランチ（`cursor/h3-kasumi-adult-0402`）。霞東本体 `kasumi-late-desk` は PR #141。このノートの Run all で本体 Drive を上書きするな
- `EPISODE = "bandai-district-short"` は 25 秒・ミッション失敗で落ちる版。`bandai-district/raw/` の暖簾・自転車・軽トラをそのまま使い、新しく描くのは理容室の 1 本だけ
- 成功時は `episode exit 0` のあと「成功。」と出る。ランタイム切断は予定どおり。`SystemExit: 0` の赤い枠は出さない

セッション名 `h3-episode`。GPU は A100。手順は `minimaxh3/episodes/README.md`。


In [ ]:
#@title 一発：上から 1・2・3 を選んで Run all（迷ったらそのまま）
EPISODE = "kasumi-late-desk-adult"  #@param {type:"string"}
#@markdown ---
#@markdown **1. つなぎ方 — 動画をどう繋げるか**
#@markdown - **カット（本ごと独立・迷ったらこれ）** … プロンプトで直したい。カメラも変えられる
#@markdown - **前の最終フレームから続ける** … 2本目以降を前クリップの最後のコマから I2V。つながり優先
#@markdown - **用意した最終フレームへ着く** … stills の jpg を最後のコマにする。jpg がある話だけ
CONNECT = "カット（本ごと独立・迷ったらこれ）"  #@param ["カット（本ごと独立・迷ったらこれ）", "前の最終フレームから続ける", "用意した最終フレームへ着く"]
#@markdown **2. カメラ**
#@markdown - **横スク（真横・全身・迷ったらこれ）** … 常に真横の 2D 横スクロール
#@markdown - **3Dアクション（引きの三人称）** … 引きの三人称。カットつなぎのとき画角が回る
CAMERA = "横スク（真横・全身・迷ったらこれ）"  #@param ["横スク（真横・全身・迷ったらこれ）", "3Dアクション（引きの三人称）"]
#@markdown **3. 画質**
#@markdown - **スピード（最速）** … 試し打ち。turbo 4step。格闘 LoRA なし
#@markdown - **バランス（迷ったらこれ）** … 普段使い。Larry 8step
#@markdown - **質（きれい・時間かかる）** … きれい優先。Larry 12step
PRESET = "バランス（迷ったらこれ）"  #@param ["スピード（最速）", "バランス（迷ったらこれ）", "質（きれい・時間かかる）"]
FRESH = False  #@param {type:"boolean"}
BRANCH = "cursor/h3-kasumi-adult-0402"  #@param {type:"string"}
print("=" * 60)
print(" H3 episode one-click:", EPISODE)
print("=" * 60)

import os, shutil, subprocess, sys, urllib.request
from pathlib import Path

from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"
COMFY_DIR = "/content/ComfyUI"
RAW = f"https://raw.githubusercontent.com/fireworker011/Research/{BRANCH}"

drive.mount("/content/drive")
os.environ["H3_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["H3_COMFY_DIR"] = COMFY_DIR
os.environ["H3_EPISODE"] = EPISODE
os.environ["H3_EPISODE_PRESET"] = PRESET
os.environ["H3_EPISODE_CAMERA"] = CAMERA
os.environ["H3_EPISODE_CONNECT"] = CONNECT
os.environ["H3_EPISODE_FRESH"] = "1" if FRESH else "0"
os.environ["H3_HELPER_BRANCH"] = BRANCH
Path(DRIVE_ROOT, "models").mkdir(parents=True, exist_ok=True)
Path(DRIVE_ROOT, "episodes", EPISODE).mkdir(parents=True, exist_ok=True)

import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。ランタイムのタイプを A100 にしてやり直してください。")
vram = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(vram, 1))
if vram < 20:
    raise SystemExit("VRAM が足りません。A100 を選んでください。")

subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-cjk", "ffmpeg"], check=False, capture_output=True)

def fetch_text(url: str, dest: Path) -> bool:
    try:
        urllib.request.urlretrieve(url, dest)
        return dest.is_file() and dest.stat().st_size > 100
    except Exception as e:
        print("fetch fail", url, e)
        return False

HELPERS = [
    "colab/h3_r2v_core.py",
    "colab/h3_motion_graphics.py",
    "colab/h3_t2v.py",
    "colab/h3_i2v_phone.py",
    "colab/h3_i2v_job.py",
    "colab/h3_i2v_runtime.py",
    "colab/h3_hud.py",
    "colab/h3_episode.py",
    "colab/h3_episode_packs.py",
    "colab/h3_episode_colab_main.py"
]
LIB = Path(DRIVE_ROOT) / "episodes" / "_lib"
LIB.mkdir(parents=True, exist_ok=True)
for rel in HELPERS:
    name = Path(rel).name
    dest = Path("/content") / name
    ok = fetch_text(f"{RAW}/{rel}", dest)
    if not ok and (LIB / name).is_file():
        shutil.copy2(LIB / name, dest)
        ok = True
    if not ok:
        raise SystemExit(f"helper missing: {name}")
    shutil.copy2(dest, LIB / name)
    print("helper", name)

sys.path.insert(0, "/content")
from h3_episode_colab_main import main

rc = main()
print("episode exit", rc)
if rc:
    raise SystemExit(rc)
print("成功。完成動画は Drive episodes/" + EPISODE + "/final/ にあります。ランタイムは停止済みです。赤い例外は出ません。")
